# Giai đoạn 2: Fine-tune ViT5 trên Dữ liệu ASR Thực tế (Wav2Vec2 Fine-tuned + KenLM -> ViT5 Rewrite)

Mục tiêu của notebook này là thực hiện trọn vẹn luồng **Fine-tune ViT5 Giai đoạn 2 (True End-to-End ASR Adaptation)** theo **ĐÚNG LUỒNG THỰC TẾ (TUYỆT ĐỐI KHÔNG FALLBACK VỀ BẤT KỲ CHẾ ĐỘ HAY MÔ HÌNH BASE NÀO)**:
1. **Giải mã Audio thực tế qua Tầng 1 (pyctcdecode + KenLM + Hotwords)**: BẮT BUỘC sử dụng mô hình acoustic Wav2Vec2 đã fine-tune của bạn kết hợp bộ giải mã `pyctcdecode` (`KenLM 4-gram` + `drugs.txt`) để nhận dạng toàn bộ tập âm thanh y khoa.
2. **Tạo Dataset Giai đoạn 2 (`train_vit5_stage2_real_asr.json`)**: Chuyển đổi các câu dự đoán ASR từ Tầng 1 thành cặp `(trans_real_asr -> target_clean_text)` chứa đúng phân phối lỗi thực tế của Wav2Vec2.
3. **Fine-tune ViT5 Giai đoạn 2**: BẮT BUỘC kế thừa huấn luyện (Warm-start) từ checkpoint ViT5 Giai đoạn 1 (`vit5_medical_rewrite_final`) để mô hình 'khắc chế' chính xác các lỗi nhận dạng âm thanh riêng biệt.

In [1]:
# 0. Cài đặt các thư viện cần thiết cho cả Wav2Vec2, pyctcdecode, KenLM và ViT5
!pip install -q -U "transformers<4.40.0" "tokenizers<0.19.0" "peft==0.10.0" "accelerate<0.29.0" protobuf datasets evaluate sentencepiece librosa
try:
    from pyctcdecode import build_ctcdecoder
except ModuleNotFoundError:
    print("⏳ Đang cài đặt pyctcdecode và kenlm...")
    !pip install -q --no-deps pyctcdecode pygtrie hypothesis rapidfuzz https://github.com/kpu/kenlm/archive/master.zip
    from pyctcdecode import build_ctcdecoder

import os
import time
import json
import torch
import librosa
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from datasets import Dataset
from transformers import (
    Wav2Vec2ForCTC,
    Wav2Vec2Processor,
    Wav2Vec2FeatureExtractor,
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    TrainerCallback
)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"[INFO] Thiết bị thực thi: {device}")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 199.1/199.1 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 94.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 92.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 290.1/290.1 kB 19.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 327.1/327.1 kB 26.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 73.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 34.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.29.

In [2]:
# 1. Nạp BẮT BUỘC mô hình Wav2Vec2 đã Fine-tune (Custom Architecture) và Trọn bộ giải mã Tầng 1 (KenLM + Hotwords)
# TUYỆT ĐỐI KHÔNG FALLBACK VỀ GREEDY HAY BASE MODEL
WAV2VEC2_PATH = "/kaggle/input/models/hdtuznn/wav2vec-finetuned/pytorch/default/1"
KENLM_BIN_PATH = "/kaggle/input/models/hdtuznn/kenlm/pytorch/default/1/kenlm_vi_medical_4gram.bin"
DRUGS_TXT_PATH = "/kaggle/input/datasets/hdtuznn/sppr-text/data/text/drugs.txt"

if not os.path.exists(WAV2VEC2_PATH):
    raise RuntimeError(f"❌ LỖI NGHIÊM TRỌNG: Không tìm thấy folder checkpoint Wav2Vec2 đã fine-tune tại '{WAV2VEC2_PATH}'.")

if not os.path.exists(KENLM_BIN_PATH):
    raise RuntimeError(f"❌ LỖI NGHIÊM TRỌNG: Không tìm thấy file KenLM tại '{KENLM_BIN_PATH}'.")

if not os.path.exists(DRUGS_TXT_PATH):
    raise RuntimeError(f"❌ LỖI NGHIÊM TRỌNG: Không tìm thấy file từ khóa thuốc tại '{DRUGS_TXT_PATH}'.")

# DEFINING CUSTOM WAV2VEC2 ARCHITECTURE MATCHING FINETUNE-WAV2VEC.IPYNB (với feature_transform 3 layers)
from transformers import Wav2Vec2PreTrainedModel, Wav2Vec2Model
from torch import nn
from transformers.modeling_outputs import CausalLMOutput
from collections import OrderedDict

_HIDDEN_STATES_START_POSITION = 2

class CustomWav2Vec2ForCTC(Wav2Vec2PreTrainedModel):
    def __init__(self, config):
        super().__init__(config)
        self.wav2vec2 = Wav2Vec2Model(config)
        self.dropout = nn.Dropout(config.final_dropout)
        self.feature_transform = nn.Sequential(OrderedDict([
            ('linear1', nn.Linear(config.hidden_size, config.hidden_size)),
            ('bn1', nn.BatchNorm1d(config.hidden_size)),
            ('activation1', nn.LeakyReLU()),
            ('drop1', nn.Dropout(config.final_dropout)),
            ('linear2', nn.Linear(config.hidden_size, config.hidden_size)),
            ('bn2', nn.BatchNorm1d(config.hidden_size)),
            ('activation2', nn.LeakyReLU()),
            ('drop2', nn.Dropout(config.final_dropout)),
            ('linear3', nn.Linear(config.hidden_size, config.hidden_size)),
            ('bn3', nn.BatchNorm1d(config.hidden_size)),
            ('activation3', nn.LeakyReLU()),
            ('drop3', nn.Dropout(config.final_dropout))
        ]))
        if config.vocab_size is None:
            raise ValueError(
                "You are trying to instantiate Wav2Vec2ForCTC with a configuration that "
                "does not define the vocabulary size of the language model head."
            )
        self.lm_head = nn.Linear(config.hidden_size, config.vocab_size)
        self.is_wav2vec_freeze = False
        self.post_init()

    def forward(
            self,
            input_values,
            attention_mask=None,
            output_attentions=None,
            output_hidden_states=None,
            return_dict=None,
            labels=None,
    ):
        return_dict = return_dict if return_dict is not None else self.config.use_return_dict
        outputs = self.wav2vec2(
            input_values,
            attention_mask=attention_mask,
            output_attentions=output_attentions,
            output_hidden_states=output_hidden_states,
            return_dict=return_dict,
        )
        hidden_states = outputs[0]
        hidden_states = self.dropout(hidden_states)
        B, T, F = hidden_states.size()
        hidden_states = hidden_states.view(B * T, F)
        hidden_states = self.feature_transform(hidden_states)
        hidden_states = hidden_states.view(B, T, F)
        logits = self.lm_head(hidden_states)
        loss = None
        if not return_dict:
            output = (logits,) + outputs[_HIDDEN_STATES_START_POSITION:]
            return ((loss,) + output) if loss is not None else output
        return CausalLMOutput(
            loss=loss, logits=logits, hidden_states=outputs.hidden_states, attentions=outputs.attentions
        )

print(f"[INFO] Đang nạp mô hình ASR Fine-tuned (Custom Architecture) từ: {WAV2VEC2_PATH}...")
asr_model = CustomWav2Vec2ForCTC.from_pretrained(WAV2VEC2_PATH).to(device)
asr_model.eval()

# FIX: Dùng Wav2Vec2CTCTokenizer trực tiếp từ vocab.json (tránh lỗi AssertionError từ tokenizer_config.json bị lỗi additional_special_tokens)
from transformers import Wav2Vec2CTCTokenizer
asr_tokenizer = Wav2Vec2CTCTokenizer(
    os.path.join(WAV2VEC2_PATH, "vocab.json"),
    unk_token="<unk>",
    pad_token="<pad>",
    word_delimiter_token="|"
)
feature_extractor = Wav2Vec2FeatureExtractor(
    feature_size=1,
    sampling_rate=16000,
    padding_value=0.0,
    do_normalize=True,
    return_attention_mask=False
)
asr_processor = Wav2Vec2Processor(feature_extractor=feature_extractor, tokenizer=asr_tokenizer)
print("✅ Nạp thành công trọng số và Processor ASR Fine-tuned!")

with open(DRUGS_TXT_PATH, "r", encoding="utf-8") as f:
    hotwords_list = [line.strip() for line in f if line.strip()]
print(f"✅ Đã nạp thành công {len(hotwords_list):,} từ khóa biệt dược vào Hotwords.")

# Xây dựng danh sách ký tự (vocab) chuẩn hóa cho pyctcdecode giống chuẩn 100% Notebook 05
vocab_dict = asr_processor.tokenizer.get_vocab()
sorted_vocab = sorted((value, key) for (key, value) in vocab_dict.items())
vocab_list = [key for (value, key) in sorted_vocab]
if "|" in vocab_dict:
    vocab_list[vocab_dict["|"]] = " "

model_vocab_size = asr_model.config.vocab_size
vocab_list = vocab_list[:model_vocab_size]
print(f"Model vocab_size: {model_vocab_size}, pyctc_vocab length: {len(vocab_list)}")

print(f"[INFO] Đang nạp bộ giải mã Tầng 1 (KenLM 4-gram) từ: {KENLM_BIN_PATH}...")
decoder_kenlm = build_ctcdecoder(
    labels=vocab_list,
    kenlm_model_path=KENLM_BIN_PATH
)
print("✅ NẠP THÀNH CÔNG TRỌN BỘ GIẢI MÃ TẦNG 1 (CUSTOM WAV2VEC2 ARCH + PYCTCDECODE) - KHÔNG FALLBACK!")

[INFO] Đang nạp mô hình ASR Fine-tuned (Custom Architecture) từ: /kaggle/input/models/hdtuznn/wav2vec-finetuned/pytorch/default/1...


Some weights of CustomWav2Vec2ForCTC were not initialized from the model checkpoint at /kaggle/input/models/hdtuznn/wav2vec-finetuned/pytorch/default/1 and are newly initialized: ['wav2vec2.encoder.pos_conv_embed.conv.parametrizations.weight.original0', 'wav2vec2.encoder.pos_conv_embed.conv.parametrizations.weight.original1']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


✅ Nạp thành công trọng số và Processor ASR Fine-tuned!
✅ Đã nạp thành công 6,069 từ khóa biệt dược vào Hotwords.
Model vocab_size: 98, pyctc_vocab length: 98
[INFO] Đang nạp bộ giải mã Tầng 1 (KenLM 4-gram) từ: /kaggle/input/models/hdtuznn/kenlm/pytorch/default/1/kenlm_vi_medical_4gram.bin...


Unigrams not provided and cannot be automatically determined from LM file (only arpa format). Decoding accuracy might be reduced.
Found entries of length > 1 in alphabet. This is unusual unless style is BPE, but the alphabet was not recognized as BPE type. Is this correct?
No known unigrams provided, decoding results might be a lot worse.


✅ NẠP THÀNH CÔNG TRỌN BỘ GIẢI MÃ TẦNG 1 (CUSTOM WAV2VEC2 ARCH + PYCTCDECODE) - KHÔNG FALLBACK!


In [3]:
# 2. Hàm nhận dạng Audio thực tế qua Tầng 1 (CHỈ DÙNG pyctcdecode + KenLM, TUYỆT ĐỐI KHÔNG FALLBACK GREEDY)
def decode_audio_to_asr_text(audio_path):
    if not os.path.exists(audio_path):
        raise FileNotFoundError(f"❌ Lỗi: Không tìm thấy file âm thanh tại {audio_path}")
    try:
        speech, _ = librosa.load(audio_path, sr=16000)
        inputs = asr_processor(speech, sampling_rate=16000, return_tensors="pt", padding=True).input_values.to(device)
        with torch.no_grad():
            logits = asr_model(inputs).logits[0].cpu().numpy()
        
        # CHỈ giải mã bằng pyctcdecode + KenLM + Hotwords (Tuyệt đối không có nhánh fallback về np.argmax greedy)
        trans = decoder_kenlm.decode(
            logits,
            beam_width=16,
            beam_prune_logp=-5.0,
            token_min_logp=-3.0,
            hotwords=hotwords_list,
            hotword_weight=15.0
        )
        return trans.strip()
    except Exception as e:
        raise RuntimeError(f"❌ Lỗi nghiêm trọng khi giải mã âm thanh {audio_path} qua Tầng 1: {e}") from e

# Hàm xây dựng dataset JSON cho ViT5 từ file CSV âm thanh
def build_vit5_dataset_from_audio_csv(csv_path, output_json_path, audio_base_dir):
    if not os.path.exists(csv_path):
        raise FileNotFoundError(f"❌ Lỗi: Không tìm thấy file CSV tại '{csv_path}'")
    
    df = pd.read_csv(csv_path)
    records = []
    print(f"\n[INFO] Bắt đầu nhận dạng Tầng 1 cho {len(df):,} file âm thanh từ {csv_path}...")
    
    for idx, row in tqdm(df.iterrows(), total=len(df), desc=f"Decoding {os.path.basename(csv_path)}"):
        rel_path = str(row["path"])
        target_text = str(row["text"]).strip()
        
        # Tự động xử lý thông minh: dù CSV có sẵn prefix 'wavs/' hay không, hoặc AUDIO_BASE_DIR lỡ thừa chữ '/wavs'
        candidate_paths = [
            rel_path,
            os.path.join(audio_base_dir, rel_path),
            os.path.join(audio_base_dir, rel_path.replace("wavs/", "") if rel_path.startswith("wavs/") else rel_path)
        ]
        full_audio_path = None
        for cp in candidate_paths:
            if os.path.exists(cp):
                full_audio_path = cp
                break
        if full_audio_path is None:
            full_audio_path = os.path.join(audio_base_dir, rel_path)
            
        real_asr_text = decode_audio_to_asr_text(full_audio_path)
        
        if real_asr_text and len(real_asr_text) >= 5:
            records.append({
                "input_text": real_asr_text,
                "target_text": target_text
            })
            
    with open(output_json_path, "w", encoding="utf-8") as f:
        json.dump(records, f, ensure_ascii=False, indent=2)
    print(f"[SUCCESS] Đã tạo thành công {len(records):,} cặp dữ liệu Real ASR -> {output_json_path}")
    
    if records:
        print("\n[MẪU DỮ LIỆU GIAI ĐOẠN 2 - ĐÚNG LUỒNG TẦNG 1]:")
        print(f"  - Input  (Real ASR + KenLM) : {records[0]['input_text']}")
        print(f"  - Target (Clean Text)       : {records[0]['target_text']}")
    return records

In [4]:
# 3. Thực thi giải mã Tầng 1 và tạo file JSON cho tập Train và Val (KHÔNG FALLBACK)
TRAIN_CSV = "/kaggle/input/datasets/hdtuznn/voice-medical/train_set/train_split.csv"
VAL_CSV = "/kaggle/input/datasets/hdtuznn/voice-medical/train_set/val_split.csv"
# CHÚ Ý: AUDIO_BASE_DIR không nên có chữ /wavs ở cuối vì trong CSV cột path đã ghi sẵn 'wavs/voice-xxxx.wav'
AUDIO_BASE_DIR = "/kaggle/input/datasets/hdtuznn/voice-medical/train_set"

# Kiểm tra ưu tiên nếu chạy trong workspace cục bộ
if os.path.exists("./finetune-wav2vec2/train_set/train_split.csv"):
    TRAIN_CSV = "./finetune-wav2vec2/train_set/train_split.csv"
    VAL_CSV = "./finetune-wav2vec2/train_set/val_split.csv"
    AUDIO_BASE_DIR = "./finetune-wav2vec2/train_set"

if not os.path.exists(TRAIN_CSV) or not os.path.exists(VAL_CSV):
    raise FileNotFoundError(f"❌ LỖI NGHIÊM TRỌNG: Không tìm thấy file train_split.csv ({TRAIN_CSV}) hoặc val_split.csv ({VAL_CSV}). BẮT BUỘC phải có dữ liệu âm thanh thực tế để fine-tune Giai đoạn 2!")

TRAIN_JSON_STAGE2 = "/kaggle/working/train_vit5_stage2_real_asr.json"
VAL_JSON_STAGE2 = "/kaggle/working/val_vit5_stage2_real_asr.json"
os.makedirs("/kaggle/working", exist_ok=True)

print("[INFO] Đang chuẩn bị tập dữ liệu huấn luyện Giai đoạn 2 qua đúng luồng Tầng 1...")
train_records = build_vit5_dataset_from_audio_csv(TRAIN_CSV, TRAIN_JSON_STAGE2, AUDIO_BASE_DIR)
val_records = build_vit5_dataset_from_audio_csv(VAL_CSV, VAL_JSON_STAGE2, AUDIO_BASE_DIR)

[INFO] Đang chuẩn bị tập dữ liệu huấn luyện Giai đoạn 2 qua đúng luồng Tầng 1...

[INFO] Bắt đầu nhận dạng Tầng 1 cho 2,727 file âm thanh từ /kaggle/input/datasets/hdtuznn/voice-medical/train_set/train_split.csv...


Decoding train_split.csv:   0%|          | 0/2727 [00:00<?, ?it/s]

[SUCCESS] Đã tạo thành công 2,724 cặp dữ liệu Real ASR -> /kaggle/working/train_vit5_stage2_real_asr.json

[MẪU DỮ LIỆU GIAI ĐOẠN 2 - ĐÚNG LUỒNG TẦNG 1]:
  - Input  (Real ASR + KenLM) : romonfoopradin gây chận nhịp tim bằng cách đặc hiệu ức chế dòng ion dòng ionày nằm ở vị trí nào trên tế bào cơ tim và nó ảnh hưởng trực tiếp đến giai đoạn nào của điện thế
  - Target (Clean Text)       : Ivabradine gây chậm nhịp tim bằng cách đặc hiệu ức chế dòng ion If. Dòng ion này nằm ở vị trí nào trên tế bào cơ tim và nó ảnh hưởng trực tiếp đến giai đoạn nào của điện thế?

[INFO] Bắt đầu nhận dạng Tầng 1 cho 120 file âm thanh từ /kaggle/input/datasets/hdtuznn/voice-medical/train_set/val_split.csv...


Decoding val_split.csv:   0%|          | 0/120 [00:00<?, ?it/s]

[SUCCESS] Đã tạo thành công 120 cặp dữ liệu Real ASR -> /kaggle/working/val_vit5_stage2_real_asr.json

[MẪU DỮ LIỆU GIAI ĐOẠN 2 - ĐÚNG LUỒNG TẦNG 1]:
  - Input  (Real ASR + KenLM) : khi bệnh nhân gặp các triệu chứng như buồn nmoltiêuchảy và trong quá trình sử dụng alcistincơ chế sinh lý no khiến các tác dụng phụ này xảy ra
  - Target (Clean Text)       : Khi bệnh nhân gặp các triệu chứng như buồn nôn, nôn, tiêu chảy và táo bón trong quá trình sử dụng L-Cystine, cơ chế sinh lý nào khiến các tác dụng phụ này xảy ra?


In [5]:
# 4. Nạp BẮT BUỘC Tokenizer và Mô hình ViT5 Giai đoạn 1 (Warm-start Fine-tuning - KHÔNG FALLBACK VỀ BASE)
VIT5_STAGE1_PATH = "/kaggle/input/models/hdtuznn/vit5-medical-rewrite-final/pytorch/default/3/vit5_medical_rewrite_final"

if not os.path.exists(VIT5_STAGE1_PATH):
    raise RuntimeError(f"❌ LỖI NGHIÊM TRỌNG: Không tìm thấy checkpoint ViT5 Giai đoạn 1 tại '{VIT5_STAGE1_PATH}'. BẮT BUỘC phải nạp mô hình ViT5 Giai đoạn 1 đã fine-tune trên text để tiếp tục fine-tune Giai đoạn 2 (TUYỆT ĐỐI KHÔNG FALLBACK về base model)!")

print(f"[INFO] Đang nạp mô hình ViT5 Giai đoạn 1 để Warm-start từ: {VIT5_STAGE1_PATH}...")
vit5_tokenizer = AutoTokenizer.from_pretrained(VIT5_STAGE1_PATH)
vit5_model = AutoModelForSeq2SeqLM.from_pretrained(VIT5_STAGE1_PATH).to(device)
vit5_model.tie_weights()
print("✅ NẠP THÀNH CÔNG ViT5 GIAI ĐOẠN 1 (Đã đồng bộ trọng số tie_weights() - KHÔNG FALLBACK)!")

MAX_INPUT_LENGTH = 128
MAX_TARGET_LENGTH = 128

def preprocess_vit5_function(examples):
    inputs = examples["input_text"]
    targets = examples["target_text"]
    model_inputs = vit5_tokenizer(inputs, max_length=MAX_INPUT_LENGTH, truncation=True)
    labels = vit5_tokenizer(text_target=targets, max_length=MAX_TARGET_LENGTH, truncation=True)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

train_dataset = Dataset.from_json(TRAIN_JSON_STAGE2)
val_dataset = Dataset.from_json(VAL_JSON_STAGE2)
print(f"[INFO] Tokenizing dữ liệu Giai đoạn 2 (Train: {len(train_dataset):,} | Val: {len(val_dataset):,})...")
tokenized_train = train_dataset.map(preprocess_vit5_function, batched=True, remove_columns=["input_text", "target_text"], desc="Tokenize Train")
tokenized_val = val_dataset.map(preprocess_vit5_function, batched=True, remove_columns=["input_text", "target_text"], desc="Tokenize Val")
data_collator = DataCollatorForSeq2Seq(tokenizer=vit5_tokenizer, model=vit5_model)
print("✅ Tokenize hoàn tất!")

[INFO] Đang nạp mô hình ViT5 Giai đoạn 1 để Warm-start từ: /kaggle/input/models/hdtuznn/vit5-medical-rewrite-final/pytorch/default/3/vit5_medical_rewrite_final...
✅ NẠP THÀNH CÔNG ViT5 GIAI ĐOẠN 1 (Đã đồng bộ trọng số tie_weights() - KHÔNG FALLBACK)!


Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

[INFO] Tokenizing dữ liệu Giai đoạn 2 (Train: 2,724 | Val: 120)...


Tokenize Train:   0%|          | 0/2724 [00:00<?, ? examples/s]

Tokenize Val:   0%|          | 0/120 [00:00<?, ? examples/s]

✅ Tokenize hoàn tất!


In [6]:
# 5. Cấu hình Callbacks và Fine-tune Giai đoạn 2 (Real ASR Adaptation)
class DetailedProgressCallback(TrainerCallback):
    def __init__(self):
        super().__init__()
        self.start_time = time.time()
        
    def on_epoch_begin(self, args, state, control, **kwargs):
        print(f"\n[INFO] ---> BẮT ĐẦU EPOCH {int(state.epoch) + 1} / {int(args.num_train_epochs)} (STAGE 2) <---")
        
    def on_epoch_end(self, args, state, control, **kwargs):
        elapsed_min = (time.time() - self.start_time) / 60.0
        epochs_done = int(state.epoch)
        total_epochs = int(args.num_train_epochs)
        est_total_min = (elapsed_min / epochs_done) * total_epochs if epochs_done > 0 else 0
        rem_min = max(0, est_total_min - elapsed_min)
        print(f"[INFO] <--- HOÀN TẤT EPOCH {epochs_done}/{total_epochs} | Đã chạy: {elapsed_min:.1f} phút | ETA còn lại: ~{rem_min:.1f} phút --->")

class GenerationStage2TestCallback(TrainerCallback):
    def on_epoch_end(self, args, state, control, model=None, **kwargs):
        if model is None:
            return
        was_training = model.training
        model.eval()
        test_q = "khi nào thì nên uống l cysn mỗi ngày có bị buồn lôn hay đau dạ dày không"
        inp = vit5_tokenizer(test_q, return_tensors="pt").to(model.device)
        with torch.no_grad():
            out = model.generate(**inp, max_length=128, num_beams=4, early_stopping=True)
        decoded = vit5_tokenizer.decode(out[0], skip_special_tokens=True)
        print(f"[STAGE 2 TEST] Epoch {int(state.epoch)} -> Decoded: {decoded}")
        if was_training:
            model.train()

OUTPUT_DIR_STAGE2 = "/kaggle/working/vit5_medical_rewrite_stage2_checkpoints"

training_args_stage2 = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR_STAGE2,
    evaluation_strategy="steps",
    eval_steps=200,
    save_strategy="steps",
    save_steps=200,
    learning_rate=1e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    weight_decay=0.01,
    save_total_limit=2,
    num_train_epochs=3,
    predict_with_generate=True,
    fp16=False,
    max_grad_norm=1.0,
    generation_max_length=128,
    logging_steps=50,
    report_to="none",
    load_best_model_at_end=False
)

stage2_trainer = Seq2SeqTrainer(
    model=vit5_model,
    args=training_args_stage2,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    tokenizer=vit5_tokenizer,
    data_collator=data_collator,
    callbacks=[DetailedProgressCallback(), GenerationStage2TestCallback()]
)

print("\n[INFO] Bắt đầu Fine-tune ViT5 Giai đoạn 2 theo đúng luồng Tầng 1 (Real ASR + KenLM)...")
stage2_trainer.train()


[INFO] Bắt đầu Fine-tune ViT5 Giai đoạn 2 theo đúng luồng Tầng 1 (Real ASR + KenLM)...


/usr/local/lib/python3.12/dist-packages/accelerate/accelerator.py:432: FutureWarning: Passing the following arguments to `Accelerator` is deprecated and will be removed in version 1.0 of Accelerate: dict_keys(['dispatch_batches', 'split_batches', 'even_batches', 'use_seedable_sampler']). Please pass an `accelerate.DataLoaderConfiguration` instead: 
dataloader_config = DataLoaderConfiguration(dispatch_batches=None, split_batches=False, even_batches=True, use_seedable_sampler=True)
  warnings.warn(



[INFO] ---> BẮT ĐẦU EPOCH 1 / 3 (STAGE 2) <---


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Step,Training Loss,Validation Loss
200,1.050800,1.539246


[INFO] <--- HOÀN TẤT EPOCH 1/3 | Đã chạy: 1.3 phút | ETA còn lại: ~2.6 phút --->
[STAGE 2 TEST] Epoch 1 -> Decoded: Khi nào thì nên uống l cysn mỗi ngày có bị buồn nôn hay đau dạ dày không?

[INFO] ---> BẮT ĐẦU EPOCH 2 / 3 (STAGE 2) <---
[INFO] <--- HOÀN TẤT EPOCH 2/3 | Đã chạy: 2.5 phút | ETA còn lại: ~1.3 phút --->
[STAGE 2 TEST] Epoch 2 -> Decoded: Khi nào thì nên uống 1 ly nước mỗi ngày có bị buồn nôn hay đau dạ dày không?

[INFO] ---> BẮT ĐẦU EPOCH 3 / 3 (STAGE 2) <---


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


[INFO] <--- HOÀN TẤT EPOCH 3/3 | Đã chạy: 3.8 phút | ETA còn lại: ~0.0 phút --->
[STAGE 2 TEST] Epoch 3 -> Decoded: Khi nào thì nên uống 1 ly nước mỗi ngày có bị buồn nôn hay đau dạ dày không?


TrainOutput(global_step=258, training_loss=1.1393019118050272, metrics={'train_runtime': 227.4982, 'train_samples_per_second': 35.921, 'train_steps_per_second': 1.134, 'total_flos': 648882692259840.0, 'train_loss': 1.1393019118050272, 'epoch': 3.0})

In [7]:
# 6. Lưu mô hình Giai đoạn 2 và Kiểm thử nhanh End-to-End Pipeline chuẩn
FINAL_MODEL_STAGE2_DIR = "/kaggle/working/vit5_medical_rewrite_stage2_final"
stage2_trainer.save_model(FINAL_MODEL_STAGE2_DIR)
vit5_model.tie_weights()
vit5_model.config.decoder_start_token_id = 0
vit5_model.config.eos_token_id = 1
vit5_model.config.pad_token_id = 0
vit5_model.save_pretrained(FINAL_MODEL_STAGE2_DIR)
vit5_tokenizer.save_pretrained(FINAL_MODEL_STAGE2_DIR)
print(f"✅ [SUCCESS] Đã lưu mô hình ViT5 Giai đoạn 2 hoàn chỉnh tại: {FINAL_MODEL_STAGE2_DIR}")

print("\n" + "="*80)
print("[SANITY CHECK] THỬ NGHIỆM ĐÚNG LUỒNG END-TO-END PIPELINE (KHÔNG FALLBACK):")
df_val_sample = pd.read_csv(VAL_CSV).head(1)
sample_rel_path = str(df_val_sample.iloc[0]["path"])
sample_target = str(df_val_sample.iloc[0]["text"])
sample_audio_path = sample_rel_path if os.path.exists(sample_rel_path) else os.path.join(AUDIO_BASE_DIR, sample_rel_path)

# 1. Tầng 1: Giải mã Audio bằng Wav2Vec2 Fine-tuned + pyctcdecode + KenLM 4-gram
trans_stage1 = decode_audio_to_asr_text(sample_audio_path)

# 2. Tầng 2: Sửa lỗi chính tả & ngữ pháp bằng ViT5 Giai đoạn 2
vit5_model.eval()
inp_text = trans_stage1
inp_tensor = vit5_tokenizer(inp_text, return_tensors="pt").to(device)
with torch.no_grad():
    out_tensor = vit5_model.generate(**inp_tensor, max_length=128, num_beams=4, early_stopping=True)
trans_stage2 = vit5_tokenizer.decode(out_tensor[0], skip_special_tokens=True)

print(f"[0. Ground Truth Target] : {sample_target}")
print(f"[1. Stage 1 ASR Output]  : {trans_stage1}")
print(f"[2. Stage 2 ViT5 Output] : {trans_stage2}")
print("="*80)

✅ [SUCCESS] Đã lưu mô hình ViT5 Giai đoạn 2 hoàn chỉnh tại: /kaggle/working/vit5_medical_rewrite_stage2_final

[SANITY CHECK] THỬ NGHIỆM ĐÚNG LUỒNG END-TO-END PIPELINE (KHÔNG FALLBACK):
[0. Ground Truth Target] : Khi bệnh nhân gặp các triệu chứng như buồn nôn, nôn, tiêu chảy và táo bón trong quá trình sử dụng L-Cystine, cơ chế sinh lý nào khiến các tác dụng phụ này xảy ra?
[1. Stage 1 ASR Output]  : khi bệnh nhân gặp các triệu chứng như buồn nmoltiêuchảy và trong quá trình sử dụng alcistincơ chế sinh lý no khiến các tác dụng phụ này xảy ra
[2. Stage 2 ViT5 Output] : Khi bệnh nhân gặp các triệu chứng như buồn nôn, tiêu chảy và trong quá trình sử dụng alcistamine, cơ chế sinh lý nào khiến các tác dụng phụ này xảy ra?
